In [ ]:
import sys
sys.path.append("../src")

from pathlib import Path
import pandas as pd

from energy_load_forecast.ingestion import load_and_resample
from energy_load_forecast.preprocessing import clean_tail, add_calendar_features, chronological_split
from energy_load_forecast.evaluation import evaluate
from energy_load_forecast.models.nbeats_model import train_nbeats, predict, series_to_darts

In [ ]:
series = load_and_resample(Path("../data/raw/entsoe/load_RO_2026-09-13_2026-09-20.csv"))
series_clean = clean_tail(series)
df = add_calendar_features(series_clean.to_frame())

train_df, val_df, test_df = chronological_split(df, train_end="2026-09-16", val_end="2026-09-18")
print("Shapes:", train_df.shape, val_df.shape, test_df.shape)

In [ ]:
# With only 72 hours in train, there are very few full 24h-in/24h-out
# training windows available -- treat this as an exploratory run, same
# caveat as LightGBM's small-data warning.
model, scaler, train_scaled = train_nbeats(
    train_df["load_mw"],
    input_chunk_length=24,
    output_chunk_length=24,
    n_epochs=50,
)

In [ ]:
val_scaled = scaler.transform(series_to_darts(val_df["load_mw"]))
history_scaled = train_scaled.append(val_scaled)

y_test = test_df["load_mw"]
forecast_series = predict(
    model, scaler, history_scaled,
    horizon=len(y_test),
    target_index=y_test.index,
)
print(forecast_series)

In [ ]:
nbeats_scores = evaluate(y_test, forecast_series)
print("N-BEATS scores:", nbeats_scores)

In [ ]:
history = pd.concat([train_df["load_mw"], val_df["load_mw"]])
horizon = len(y_test)

naive_pred = pd.Series(history.iloc[-1], index=y_test.index, name="forecast")

season_length = 24
seasonal_block = history.iloc[-season_length:]
n_tiles = -(-horizon // season_length)
seasonal_values = list(seasonal_block.to_numpy()) * n_tiles
seasonal_pred = pd.Series(seasonal_values[:horizon], index=y_test.index, name="forecast")

naive_scores = evaluate(y_test, naive_pred)
seasonal_scores = evaluate(y_test, seasonal_pred)

comparison = pd.DataFrame(
    [naive_scores, seasonal_scores, nbeats_scores],
    index=["Naive", "Seasonal Naive", "N-BEATS"],
)
print("Comparison:\n", comparison)